Carga de DataSET de Bronce

In [0]:
# Cargar datasets desde Bronze

base_path = "dbfs:/Volumes/workspace/default/churn_b2b/Datos"

df_empresas = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/empresas.csv")
df_facturacion = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/facturacion.csv")
df_consumo = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/consumo.csv")
df_servicios = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/servicios.csv")
df_tickets = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/tickets.csv")
df_eventos = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/eventos_red.csv")
df_churn = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/churn.csv")

print("Todos los datasets fueron cargados correctamente.")

Todos los datasets fueron cargados correctamente.


Celda 1 - Importaciones

In [0]:
from pyspark.sql.functions import col, count, when

Celda 2 - Verificar esquemas

In [0]:
datasets = {
    "Empresas": df_empresas,
    "Facturación": df_facturacion,
    "Consumo": df_consumo,
    "Servicios": df_servicios,
    "Tickets": df_tickets,
    "Eventos": df_eventos,
    "Churn": df_churn
}

for nombre, df in datasets.items():
    print("=" * 60)
    print(nombre)
    df.printSchema()

Empresas
root
 |-- company_id: integer (nullable = true)
 |-- ruc: long (nullable = true)
 |-- razon_social: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- region: string (nullable = true)
 |-- empleados: integer (nullable = true)
 |-- segmento: string (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- ejecutivo_comercial: string (nullable = true)
 |-- nps: integer (nullable = true)
 |-- csat: double (nullable = true)

Facturación
root
 |-- factura_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- periodo: integer (nullable = true)
 |-- monto_facturado: double (nullable = true)
 |-- dias_mora: integer (nullable = true)
 |-- deuda_actual: double (nullable = true)

Consumo
root
 |-- consumo_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- periodo: integer (nullable = true)
 |-- trafico_gb: double (nullable = true)
 |-- ancho_banda_promedio: double (nullable = true)
 |-- pico_maximo_mbps: double (nul

Celda 3 - Verificar duplicados

In [0]:
for nombre, df in datasets.items():
    total = df.count()
    unicos = df.dropDuplicates().count()

    print(f"{nombre}")
    print(f"Total: {total}")
    print(f"Duplicados: {total - unicos}")
    print("-" * 40)

Empresas
Total: 5000
Duplicados: 0
----------------------------------------
Facturación
Total: 120000
Duplicados: 0
----------------------------------------
Consumo
Total: 120000
Duplicados: 0
----------------------------------------
Servicios
Total: 15140
Duplicados: 0
----------------------------------------
Tickets
Total: 102980
Duplicados: 0
----------------------------------------
Eventos
Total: 273378
Duplicados: 0
----------------------------------------
Churn
Total: 5000
Duplicados: 0
----------------------------------------


Celda 4 - Eliminar duplicados

In [0]:
df_empresas = df_empresas.dropDuplicates()
df_facturacion = df_facturacion.dropDuplicates()
df_consumo = df_consumo.dropDuplicates()
df_servicios = df_servicios.dropDuplicates()
df_tickets = df_tickets.dropDuplicates()
df_eventos = df_eventos.dropDuplicates()
df_churn = df_churn.dropDuplicates()

Celda 5 - Verificar valores nulos

In [0]:
for nombre, df in datasets.items():
    print("=" * 60)
    print(nombre)

    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()

Empresas
+----------+---+------------+------+------+---------+--------+----------+-------------------+---+----+
|company_id|ruc|razon_social|sector|region|empleados|segmento|fecha_alta|ejecutivo_comercial|nps|csat|
+----------+---+------------+------+------+---------+--------+----------+-------------------+---+----+
|         0|  0|           0|     0|     0|        0|       0|         0|                  0|  0|   0|
+----------+---+------------+------+------+---------+--------+----------+-------------------+---+----+

Facturación
+----------+----------+-------+---------------+---------+------------+
|factura_id|company_id|periodo|monto_facturado|dias_mora|deuda_actual|
+----------+----------+-------+---------------+---------+------------+
|         0|         0|      0|              0|        0|           0|
+----------+----------+-------+---------------+---------+------------+

Consumo
+----------+----------+-------+----------+--------------------+----------------+
|consumo_id|compan

Celda 6 - Crear vistas temporales

In [0]:
df_empresas.createOrReplaceTempView("bronze_empresas")
df_facturacion.createOrReplaceTempView("bronze_facturacion")
df_consumo.createOrReplaceTempView("bronze_consumo")
df_servicios.createOrReplaceTempView("bronze_servicios")
df_tickets.createOrReplaceTempView("bronze_tickets")
df_eventos.createOrReplaceTempView("bronze_eventos")
df_churn.createOrReplaceTempView("bronze_churn")

Celda 7 - Verificar las vistas

In [0]:
spark.sql("SELECT COUNT(*) FROM bronze_empresas").show()
spark.sql("SELECT COUNT(*) FROM bronze_facturacion").show()
spark.sql("SELECT COUNT(*) FROM bronze_consumo").show()
spark.sql("SELECT COUNT(*) FROM bronze_servicios").show()
spark.sql("SELECT COUNT(*) FROM bronze_tickets").show()
spark.sql("SELECT COUNT(*) FROM bronze_eventos").show()
spark.sql("SELECT COUNT(*) FROM bronze_churn").show()

+--------+
|COUNT(*)|
+--------+
|    5000|
+--------+

+--------+
|COUNT(*)|
+--------+
|  120000|
+--------+

+--------+
|COUNT(*)|
+--------+
|  120000|
+--------+

+--------+
|COUNT(*)|
+--------+
|   15140|
+--------+

+--------+
|COUNT(*)|
+--------+
|  102980|
+--------+

+--------+
|COUNT(*)|
+--------+
|  273378|
+--------+

+--------+
|COUNT(*)|
+--------+
|    5000|
+--------+



Guardar Bronze

In [0]:
df_empresas.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/empresas")

df_facturacion.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/facturacion")

df_consumo.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/consumo")

df_servicios.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/servicios")

df_tickets.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/tickets")

df_eventos.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/eventos_red")

df_churn.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/churn_b2b/Bronze/churn")